# Embedding and Retrieval Experiments

Compare multilingual text-embedding models using the same production chunks and grounded retrieval cases

## 1. Setup

In [1]:
import gc
import importlib
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this experiment.")

print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce MX570 A


## 2. Select documents and candidate models

In [2]:
SUBSET_FILES = None

MODEL_CONFIGS = {
    "multilingual_e5_base": {
        "enabled": True,
        "model_name": "intfloat/multilingual-e5-base",
        "batch_size": 8,
    },
    "paraphrase_multilingual_minilm": {
        "enabled": True,
        "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 16,
    },
    "bge_m3": {
        "enabled": True,
        "model_name": "BAAI/bge-m3",
        "batch_size": 1,
    },
}

enabled_models = {
    key: value for key, value in MODEL_CONFIGS.items() if value["enabled"]
}
print("Evaluation scope: all documents represented by test cases")
print("Models that will run:", list(enabled_models))

Evaluation scope: all documents represented by test cases
Models that will run: ['multilingual_e5_base', 'paraphrase_multilingual_minilm', 'bge_m3']


## 3. Load production chunks and evaluation cases

In [3]:
pipeline_module = importlib.reload(importlib.import_module("doc_rag.pipeline"))
embedding_module = importlib.reload(importlib.import_module("doc_rag.5_embeddings"))

pipeline_result = pipeline_module.run_pipeline(
    PROJECT_ROOT / "docs",
    recursive=True,
    extract_images=False,
    create_embeddings=False,
    create_index=False,
)

all_chunks = pipeline_result["chunks"]
CASES_FILE = (
    PROJECT_ROOT / "data" / "rag" / "experiments"
    / "chunking_evaluation" / "document_chunking_cases.json"
)
with CASES_FILE.open(encoding="utf-8") as file:
    all_cases = json.load(file)["cases"]
SUBSET_FILES = sorted({case["expected_file"] for case in all_cases})
subset_chunks = (
    all_chunks[all_chunks["file_name"].isin(SUBSET_FILES)]
    .reset_index(drop=True)
)
missing_subset_files = set(SUBSET_FILES) - set(subset_chunks["file_name"])
if missing_subset_files:
    raise ValueError(f"Evaluation PDFs were not found: {sorted(missing_subset_files)}")
evaluation_cases = all_cases
if not evaluation_cases:
    raise ValueError("No evaluation cases match the selected documents.")

print("All production chunks:", len(all_chunks))
print("Subset chunks:", len(subset_chunks))
print("Evaluation cases:", len(evaluation_cases))
display(subset_chunks.groupby("file_name").size().rename("chunks"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

All production chunks: 775
Subset chunks: 775
Evaluation cases: 42


file_name
Boho_Wedding.pdf                         61
Celestial_Wedding.pdf                    14
Checklists.pdf                           17
Classic_Wedding.pdf                      20
Enchanted_Forest_Wedding.pdf             44
Garden_Wedding.pdf                       30
Guideline_Sustainable_Event.pdf          70
Modern_Wedding.pdf                       61
Moody_Wedding.pdf                        20
Organiser_un_evenement_deAaZ.pdf         82
Romantic_Wedding.pdf                     23
Rustic_Wedding.pdf                       61
Tropical_Wedding.pdf                     20
Vintage_Wedding.pdf                      19
Wedding_Centerpieces.pdf                 22
Wedding_Colors.pdf                       19
Wedding_Themes.pdf                       21
Wedding_Trends.pdf                       64
Whimsical_Wedding.pdf                    21
تقاليد الزفاف في الثقافات العربية.pdf    86
Name: chunks, dtype: int64

## 4. Embed and evaluate each model

In [4]:
EXPERIMENT_EMBEDDING_ROOT = (
    PROJECT_ROOT / "data" / "rag" / "experiments"
    / "embedding_model_comparison" / "embeddings"
)
TOP_K = 5
model_metric_rows = []
case_result_rows = []

for model_key, config in enabled_models.items():
    model_name = config["model_name"]
    batch_size = config["batch_size"]
    
    print("MODEL:", model_key, "-", model_name)

    torch.cuda.empty_cache()
    model = embedding_module.load_embedding_model(
        model_name,
        device="cuda",
        local_files_only=True,
        model_kwargs={"torch_dtype": torch.float16},
    )

    started = perf_counter()
    embedding_result = embedding_module.get_or_create_chunk_embeddings(
        subset_chunks,
        model_name=model_name,
        embedding_root=EXPERIMENT_EMBEDDING_ROOT,
        model=model,
        batch_size=batch_size,
    )
    chunk_embedding_seconds = perf_counter() - started
    chunk_embeddings = embedding_result["embeddings"]

    queries = [case["query"] for case in evaluation_cases]
    started = perf_counter()
    query_embeddings = embedding_module.embed_queries(
        queries, model=model, model_name=model_name, batch_size=batch_size
    )
    query_embedding_seconds = perf_counter() - started
    scores, positions = embedding_module.cosine_search(
        query_embeddings, chunk_embeddings, top_k=TOP_K
    )

    ranks = []
    for case_index, case in enumerate(evaluation_cases):
        expected_pages = set(case["expected_pages"])
        first_relevant_rank = None
        top_results = []
        for rank, position in enumerate(positions[case_index], start=1):
            chunk = subset_chunks.iloc[int(position)]
            relevant = (
                chunk["file_name"] == case["expected_file"]
                and int(chunk["page_number"]) in expected_pages
            )
            if relevant and first_relevant_rank is None:
                first_relevant_rank = rank
            top_results.append({
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "file_name": chunk["file_name"],
                "page_number": int(chunk["page_number"]),
                "score": round(float(scores[case_index, rank - 1]), 6),
                "relevant": bool(relevant),
            })
        ranks.append(first_relevant_rank)
        case_result_rows.append({
            "model_key": model_key,
            "model_name": model_name,
            "case_id": case["id"],
            "query": case["query"],
            "expected_file": case["expected_file"],
            "expected_pages": case["expected_pages"],
            "first_relevant_rank": first_relevant_rank,
            "top_5": top_results,
        })

    reciprocal_ranks = [0 if rank is None else 1 / rank for rank in ranks]
    model_metric_rows.append({
        "model_key": model_key,
        "model_name": model_name,
        "dimension": int(chunk_embeddings.shape[1]),
        "chunks": len(subset_chunks),
        "cases": len(evaluation_cases),
        "recall_at_1": round(np.mean([r is not None and r <= 1 for r in ranks]), 4),
        "recall_at_3": round(np.mean([r is not None and r <= 3 for r in ranks]), 4),
        "recall_at_5": round(np.mean([r is not None and r <= 5 for r in ranks]), 4),
        "mrr_at_5": round(float(np.mean(reciprocal_ranks)), 4),
        "chunk_embedding_seconds": round(chunk_embedding_seconds, 3),
        "query_embedding_seconds": round(query_embedding_seconds, 3),
        "cache_hit": bool(embedding_result["cache_hit"]),
        "embedding_file_bytes": embedding_result["embeddings_file"].stat().st_size,
    })

    del model, chunk_embeddings, query_embeddings, embedding_result
    gc.collect()
    torch.cuda.empty_cache()

model_metrics = pd.DataFrame(model_metric_rows).set_index("model_key")
case_results = pd.DataFrame(case_result_rows)
display(model_metrics.sort_values(
    ["recall_at_5", "mrr_at_5", "recall_at_1"], ascending=False
))

MODEL: multilingual_e5_base - intfloat/multilingual-e5-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODEL: paraphrase_multilingual_minilm - sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODEL: bge_m3 - BAAI/bge-m3


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,model_name,dimension,chunks,cases,recall_at_1,recall_at_3,recall_at_5,mrr_at_5,chunk_embedding_seconds,query_embedding_seconds,cache_hit,embedding_file_bytes
model_key,,,,,,,,,,,,
bge_m3,BAAI/bge-m3,1024,775,42,0.5952,0.8095,0.8333,0.6992,0.128,2.052,True,3174528
multilingual_e5_base,intfloat/multilingual-e5-base,768,775,42,0.5238,0.7857,0.8333,0.6548,0.013,0.751,True,2380928
paraphrase_multilingual_minilm,sentence-transformers/paraphrase-multilingual-...,384,775,42,0.2857,0.3571,0.4286,0.3290,0.021,0.142,True,1190528


## 5. Inspect failures and model disagreements

In [5]:
failures = case_results[case_results["first_relevant_rank"].isna()]
display(failures[[
    "model_key", "case_id", "query", "expected_file", "expected_pages"
]])

rank_comparison = case_results.pivot(
    index=["case_id", "query", "expected_file"],
    columns="model_key",
    values="first_relevant_rank",
).reset_index()
display(rank_comparison)

,model_key,case_id,query,expected_file,expected_pages
7,multilingual_e5_base,modern_acrylic,How can acrylic menus and black-and-white colo...,Modern_Wedding.pdf,[15]
21,multilingual_e5_base,organiser_role,What does an event organizer actually do when ...,Organiser_un_evenement_deAaZ.pdf,"[5, 6]"
22,multilingual_e5_base,organiser_logistics,"Who is responsible for managing suppliers, equ...",Organiser_un_evenement_deAaZ.pdf,[11]
24,multilingual_e5_base,organiser_translation,Si certains participants ne comprennent pas la...,Organiser_un_evenement_deAaZ.pdf,[17]
25,multilingual_e5_base,organiser_agency,Quel est le rôle d’une agence événementielle e...,Organiser_un_evenement_deAaZ.pdf,[18]
28,multilingual_e5_base,checklists_energy_water,What energy and water savings measures should ...,Checklists.pdf,[5]
31,multilingual_e5_base,guideline_minimum_criteria,I want to organize a sustainable event but I c...,Guideline_Sustainable_Event.pdf,[3]
43,paraphrase_multilingual_minilm,boho_macrame,How can macrame or fringe be used in a bohemia...,Boho_Wedding.pdf,"[8, 18]"
46,paraphrase_multilingual_minilm,classic_timeless,What classic wedding ideas stay stylish across...,Classic_Wedding.pdf,"[1, 2]"
49,paraphrase_multilingual_minilm,modern_acrylic,How can acrylic menus and black-and-white colo...,Modern_Wedding.pdf,[15]


model_key,case_id,query,expected_file,bge_m3,multilingual_e5_base,paraphrase_multilingual_minilm
0,arabic_egyptian_stages,أخطط لزفاف مصري والعميل يريد الإبقاء على المرا...,تقاليد الزفاف في الثقافات العربية.pdf,1.0,1.0,NaN
1,arabic_emirati_preparation,أنظّم زفافاً إماراتياً. ما الاستعدادات التقليد...,تقاليد الزفاف في الثقافات العربية.pdf,1.0,1.0,NaN
2,arabic_lebanese_customs,أخطط لزفاف لبناني وأريد أن يبدو تقليدياً دون أ...,تقاليد الزفاف في الثقافات العربية.pdf,1.0,1.0,NaN
3,arabic_lebanese_lesser_known,عم حضّر عرس لبناني والعميل بده عادات قديمة ومم...,تقاليد الزفاف في الثقافات العربية.pdf,2.0,1.0,NaN
4,arabic_moroccan_entrance,أخطط لزفاف مغربي وأريد أن يكون دخول العروس تقل...,تقاليد الزفاف في الثقافات العربية.pdf,2.0,2.0,NaN
5,arabic_moroccan_henna,العميل يريد ليلة حناء مغربية أصيلة، وليس مجرد ...,تقاليد الزفاف في الثقافات العربية.pdf,3.0,2.0,NaN
6,arabic_najdi_no_henna,أخطط لزفاف نجدي وأريد تجنب إضافة عادات لا تنتم...,تقاليد الزفاف في الثقافات العربية.pdf,1.0,2.0,NaN
7,beach_colors,What beach wedding colors can be used beyond o...,Wedding_Colors.pdf,1.0,1.0,1.0
8,boho_floral_crown,What can a boho bride wear instead of a veil f...,Boho_Wedding.pdf,1.0,1.0,1.0
9,boho_macrame,How can macrame or fringe be used in a bohemia...,Boho_Wedding.pdf,1.0,3.0,NaN


## 6. Save the evaluation and select the best model

In [6]:
ranked_models = model_metrics.reset_index().sort_values(
    ["recall_at_5", "mrr_at_5", "recall_at_1"],
    ascending=False,
).reset_index(drop=True)
RECOMMENDED_MODEL_KEY = ranked_models.iloc[0]["model_key"]
RECOMMENDED_MODEL_NAME = ranked_models.iloc[0]["model_name"]

RESULTS_FILE = (
    PROJECT_ROOT / "data" / "rag" / "experiments"
    / "embedding_model_comparison" / "results.json"
)
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "schema_version": 1,
    "subset_files": SUBSET_FILES,
    "chunking_strategy": pipeline_result["chunking_strategy"],
    "chunking_options": pipeline_result["chunking_options"],
    "model_configs": enabled_models,
    "recommended_model_key": RECOMMENDED_MODEL_KEY,
    "recommended_model_name": RECOMMENDED_MODEL_NAME,
    "metrics": model_metrics.reset_index().to_dict(orient="records"),
    "case_results": case_result_rows,
}
with RESULTS_FILE.open("w", encoding="utf-8") as file:
    json.dump(payload, file, ensure_ascii=False, indent=2)

print("RECOMMENDED MODEL:", RECOMMENDED_MODEL_NAME)
print("Results saved to:", RESULTS_FILE)
display(ranked_models)

RECOMMENDED MODEL: BAAI/bge-m3
Results saved to: c:\Users\User\Desktop\inmind\gatherly_rag\data\rag\experiments\embedding_model_comparison\results.json


,model_key,model_name,dimension,chunks,cases,recall_at_1,recall_at_3,recall_at_5,mrr_at_5,chunk_embedding_seconds,query_embedding_seconds,cache_hit,embedding_file_bytes
0,bge_m3,BAAI/bge-m3,1024,775,42,0.5952,0.8095,0.8333,0.6992,0.128,2.052,True,3174528
1,multilingual_e5_base,intfloat/multilingual-e5-base,768,775,42,0.5238,0.7857,0.8333,0.6548,0.013,0.751,True,2380928
2,paraphrase_multilingual_minilm,sentence-transformers/paraphrase-multilingual-...,384,775,42,0.2857,0.3571,0.4286,0.3290,0.021,0.142,True,1190528


## 7. Release GPU memory

In [7]:
# Models are already released after each comparison; this clears final references.
for variable_name in [
    "model",
    "chunk_embeddings",
    "query_embeddings",
    "embedding_result",
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print("GPU allocated after cleanup (MB):", round(torch.cuda.memory_allocated() / 1024**2, 2))
print("GPU reserved after cleanup (MB):", round(torch.cuda.memory_reserved() / 1024**2, 2))
print("Embedding experiment complete and GPU model memory released.")

GPU allocated after cleanup (MB): 9.12
GPU reserved after cleanup (MB): 22.0
Embedding experiment complete and GPU model memory released.
